In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModel
import pickle
from typing import List, Dict
from tqdm import tqdm

def encode_texts_to_cls_embeddings(
    ids: List[str],
    texts: List[str],
    model_name: str = "bert-base-uncased",
    output_path: str = "id_to_embedding.pkl",
    device: str = None
) -> None:
    """
    与えられたIDとテキストを、指定されたTransformerモデルでCLS埋め込みに変換し、辞書に保存してPickleファイルとして出力する。

    Parameters:
        ids (List[str]): テキストごとのIDリスト
        texts (List[str]): 対応する文章リスト
        model_name (str): 利用するモデル名（例: "bert-base-uncased"）
        output_path (str): 出力するPickleファイルパス
        device (str): 使用するデバイス（'cuda'または'cpu'）。Noneなら自動判定。
    """

    assert len(ids) == len(texts), "IDsとtextsの長さが一致していません。"

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    id_to_embedding: Dict[str, torch.Tensor] = {}

    with torch.no_grad():
        for i, (id_, text) in enumerate(tqdm(zip(ids, texts))):
            inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
            outputs = model(**inputs)
            cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze(0).cpu()  # CLSトークンのベクトル

            id_to_embedding[id_] = cls_embedding

    # tensorをnumpyに変換して保存（オプションでtensorのままでも可）
    id_to_numpy = {k: v.numpy() for k, v in id_to_embedding.items()}

    with open(output_path, "wb") as f:
        pickle.dump(id_to_numpy, f)

    print(f"保存完了: {output_path}")

In [32]:
from dvrl.dataset import EssayDataset
import numpy as np

dataset = EssayDataset(
    main_file='../data/training_set_rel3.xlsx',
    feature_file='../data/hand_crafted_v3.csv',
    readability_file='../data/readability_features.csv'
)

source_data, target_data = dataset.cross_prompt_split(
    target_prompt_set=1,
    add_pos=False
)

In [33]:
ids = np.concatenate([target_data['essay_id'], source_data['essay_id']])
texts = np.concatenate([target_data['essay'], source_data['essay']])
model_name = 'microsoft/deberta-v3-large'

encode_texts_to_cls_embeddings(
    ids.tolist(), texts.tolist(),
    model_name=model_name,
    output_path=f"../outputs/embedding/{os.path.basename(model_name)}.pkl",
    device="cuda",
)

/home/ito/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
/home/ito/.local/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:470: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embedding

保存完了: ../outputs/embedding/deberta-v3-large.pkl


In [34]:
with open('../outputs/embedding/deberta-v3-large.pkl', 'rb') as f:
    data = pickle.load(f)

In [36]:
data

{1: array([ 5.8205165e-02, -8.2956173e-02,  1.7743178e-02, ...,
         9.7262353e-04, -5.7996387e+00,  7.3884167e-02], dtype=float32),
 2: array([ 0.05266773, -0.0337947 , -0.01497474, ...,  0.01041585,
        -5.795081  ,  0.05186283], dtype=float32),
 3: array([ 2.1239758e-02, -1.1526563e-01,  8.8912295e-03, ...,
         3.3989975e-03, -5.8038049e+00,  8.9093685e-02], dtype=float32),
 4: array([ 0.07526371, -0.03758738,  0.0248458 , ..., -0.01106146,
        -5.8089724 ,  0.02081241], dtype=float32),
 5: array([ 7.2444528e-02, -4.9025167e-02, -1.0863581e-02, ...,
        -7.9990056e-04, -5.8040900e+00,  2.0965833e-02], dtype=float32),
 6: array([ 0.03712868, -0.04185134, -0.0266336 , ...,  0.01685499,
        -5.8112836 ,  0.04175467], dtype=float32),
 7: array([ 0.10218118, -0.03954424,  0.04386939, ..., -0.01287137,
        -5.805042  ,  0.03586499], dtype=float32),
 8: array([ 8.0688626e-02, -4.6589568e-02,  2.3404737e-03, ...,
        -7.1867895e-03, -5.8042994e+00,  4.021807